# Networks, 1956: Dijkstra's Shortest Path

In 1956, Edsger Dijkstra designed the shortest-path algorithm in about twenty minutes at a café on the Leidseplein in Amsterdam, as a demo problem for the unveiling of the ARMAC computer. He published it as three pages in *Numerische Mathematik* in 1959. Within a decade it was the core of every network routing protocol on earth.

This notebook builds a weighted graph, runs Dijkstra, and examines where the algorithm breaks.

**Accompanies the lesson:** [`/lessons/graph-theory/networks`](https://lutchet.netlify.app/lessons/graph-theory/networks)

In [1]:
import sys
sys.path.insert(0, '..')

from src.graph_theory.weighted import WeightedGraph
from src.graph_theory.dijkstra import dijkstra, shortest_path

## 1. A Dutch road graph

Ten cities, fourteen roads, approximate driving distances in kilometres. This is a small version of the 64-city graph Dijkstra used for the ARMAC demo.

In [2]:
g = WeightedGraph()

roads = [
    ('AMS', 'HAA',  20), ('AMS', 'UTR',  45), ('HAA', 'DHG',  45),
    ('DHG', 'ROT',  25), ('ROT', 'UTR',  60), ('UTR', 'ARN',  65),
    ('UTR', 'EIN', 100), ('ROT', 'EIN', 110), ('ARN', 'ZWO',  65),
    ('ARN', 'EIN',  90), ('ZWO', 'GRO', 100), ('AMS', 'ZWO', 120),
    ('EIN', 'MAA',  85), ('ARN', 'MAA', 180),
]

names = {
    'AMS': 'Amsterdam', 'HAA': 'Haarlem',   'UTR': 'Utrecht',
    'ROT': 'Rotterdam', 'DHG': 'The Hague', 'ARN': 'Arnhem',
    'ZWO': 'Zwolle',    'GRO': 'Groningen', 'EIN': 'Eindhoven',
    'MAA': 'Maastricht',
}

for u, v, km in roads:
    g.add_edge(u, v, km)

print('Cities:', len(g.nodes))
print('Roads: ', len(g.edges))

Cities: 10
Roads:  14


## 2. Shortest paths from Rotterdam

`dijkstra` returns two dictionaries: distances from the source to every node, and the predecessor of each node on the shortest-path tree.

In [3]:
distances, predecessors = dijkstra(g, 'ROT')

for city_id in sorted(distances, key=lambda c: distances[c]):
    print(f'  Rotterdam -> {names[city_id]:12s}  {distances[city_id]:>5.0f} km')

  Rotterdam -> Rotterdam         0 km
  Rotterdam -> The Hague        25 km
  Rotterdam -> Utrecht          60 km
  Rotterdam -> Haarlem          70 km
  Rotterdam -> Amsterdam        90 km
  Rotterdam -> Eindhoven       110 km
  Rotterdam -> Arnhem          125 km
  Rotterdam -> Zwolle          190 km
  Rotterdam -> Maastricht      195 km
  Rotterdam -> Groningen       290 km


## 3. Recover a path

The predecessor map lets us reconstruct any shortest path by walking backwards from the target.

In [4]:
path = shortest_path(predecessors, 'ROT', 'GRO')
total = distances['GRO']

print(f'Rotterdam to Groningen: {total:.0f} km')
print('Route:', ' -> '.join(names[p] for p in path))

Rotterdam to Groningen: 290 km
Route: Rotterdam -> Utrecht -> Arnhem -> Zwolle -> Groningen


## 4. When greedy fails: negative weights

Dijkstra's greedy choice relies on the invariant that once a node is settled, no future relaxation can improve its distance. A negative edge can break this: a longer-looking path might gain a discount further along. `WeightedGraph.add_edge` rejects negative weights at insert time, which catches the mistake before it can corrupt the algorithm.

In [5]:
broken = WeightedGraph()
try:
    broken.add_edge('X', 'Y', -3.0)
except ValueError as e:
    print('Rejected:', e)

Rejected: Edge ('X', 'Y') has negative weight -3.0; Dijkstra requires non-negative weights.


For graphs that genuinely need negative edges (currency arbitrage, certain routing-policy computations), the algorithm you want is Bellman-Ford (1958). It is slower, but it does not make the greedy assumption. We leave it as an exercise.

## 5. A tiny ARPANET

Dijkstra's algorithm became the core of network routing a decade after it was published. Here is a toy version: five routers, weights proportional to link latency. Run Dijkstra from router A to find the shortest-latency path to every other router.

In [6]:
net = WeightedGraph()
net.add_edge('A', 'B', 5)
net.add_edge('A', 'C', 2)
net.add_edge('B', 'C', 1)
net.add_edge('B', 'D', 3)
net.add_edge('C', 'D', 6)
net.add_edge('D', 'E', 2)
net.add_edge('C', 'E', 10)

d, p = dijkstra(net, 'A')
for node in 'ABCDE':
    route = shortest_path(p, 'A', node)
    print(f'  A -> {node}: latency {d[node]:>2} via {" -> ".join(route)}')

  A -> A: latency 0.0 via A
  A -> B: latency 3.0 via A -> C -> B
  A -> C: latency 2.0 via A -> C
  A -> D: latency 6.0 via A -> C -> B -> D
  A -> E: latency 8.0 via A -> C -> B -> D -> E


## 6. Your turn

A few things to try:

- Close the road between Rotterdam and Utrecht by increasing its weight to a very large value. How does the shortest path from Rotterdam to Groningen change?
- Pick two cities and compare the shortest-path length against the path with the fewest edges. When does hop-count disagree with distance?
- Add a direct road from The Hague to Groningen with weight 180 km. How many of the existing shortest paths shift?

The next story in this topic leaves single-source shortest paths behind and asks a different question: given a network, how much can flow through it from a source to a sink, and which bottlenecks decide the answer? That is Ford and Fulkerson, 1956, motivated by a Cold War study of the Soviet rail network.